# 07 — Frontend (React + TypeScript)

Vite + React 18 + TypeScript + Recharts. Pages: Overview (dashboard KPIs + 7 charts), Review Analyzer (single-review pipeline), Batch Upload, Customers/Sellers/Products/Geography (analytics), Model Info.

`src/api/` centralizes all HTTP calls; `src/types/` mirrors backend Pydantic schemas field-for-field (the TypeScript side of `shared/api_contract.json`); `src/hooks/` wraps loading/error state; `src/pages/` + `src/components/` are pure presentation -- no ML logic in the frontend. See `FRONTEND_INTEGRATION.md`.

In [ ]:
import os
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Walk upward from wherever this notebook actually lives to find the real
    project root (the folder containing both backend/app/ and data/), so every
    relative path used below resolves correctly regardless of which folder
    this notebook is opened from."""
    for candidate in [start, *start.parents]:
        if (candidate / "backend" / "app").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError(
        "Could not locate the Baseera project root (a folder containing both "
        "backend/app/ and data/) above this notebook's location."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "backend"))
print("Project root:", PROJECT_ROOT)


In [ ]:
from pathlib import Path

src = PROJECT_ROOT / "frontend" / "src"
for sub in ["api", "pages", "components", "hooks", "types"]:
    p = src / sub
    if p.is_dir():
        names = sorted(x.name for x in p.iterdir())
        print(f"{sub:12s} ({len(names)}):", names[:8], "..." if len(names) > 8 else "")


```ts
// frontend/src/api/client.ts -- idempotency key generated client-side, sent on retry too
export async function apiPost<T>(url: string, body: unknown): Promise<T> {
  const idempotencyKey = crypto.randomUUID();
  const headers = { "Content-Type": "application/json", "Idempotency-Key": idempotencyKey };
  return withColdStartRetry(() => axios.post<T>(url, body, { headers }));
}
```

**A real UX bug fixed**: the fake-review verdict badge used to show a confident-looking "LABEL_1 (assumed 'fake')" even when the underlying model's verdict had flipped under a paraphrase probe -- the frontend now checks `verdict === "UNCERTAIN"` and shows a genuinely different, unstyled state instead of coloring an unreliable answer as if it were reliable (`FakeCheckBadge.tsx`).